# Lip-sync only — GPU (free Colab)

You already have a **dubbed video** (original picture + the new English audio, e.g.
`fc4BozJTvc4.en.mp4`). This notebook does **only Wav2Lip** on a GPU: it re-renders the
mouth so the lips match the dubbed audio. ~1 minute on a T4 vs many minutes on CPU.

**Runtime → Change runtime type → GPU (T4)**, then Runtime → Run all.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

## 2. Install Wav2Lip in its own venv (isolated, GPU torch) + checkpoints
Takes a couple of minutes (downloads a CUDA torch wheel + the model). The two
checkpoint URLs below are known-good; if one ever 404s, swap it for another mirror.

In [ ]:
%%bash
set -e
cd /content
[ -d Wav2Lip ] || git clone -q https://github.com/Rudrabha/Wav2Lip
cd Wav2Lip
python3 -m venv .venv
./.venv/bin/pip -q install --upgrade pip
./.venv/bin/pip -q install torch torchvision numpy==1.23.5 librosa==0.9.2 \
    numba==0.58.1 opencv-python-headless 'scipy<1.13' tqdm 'setuptools<80'
mkdir -p checkpoints face_detection/detection/sfd
wget -q -O face_detection/detection/sfd/s3fd.pth \
  https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth
wget -q -O checkpoints/wav2lip_gan.pth \
  https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth
echo '--- setup done ---'
ls -lh checkpoints/wav2lip_gan.pth face_detection/detection/sfd/s3fd.pth

## 3. Upload your dubbed video
Pick the dubbed MP4 from your machine (the one with the English audio, e.g.
`data_final/output/fc4BozJTvc4.en.mp4`). Wav2Lip takes both the picture and the
target audio from this single file.

In [ ]:
from google.colab import files
import shutil
up = files.upload()
name = list(up.keys())[0]
shutil.move(name, '/content/input.mp4')
print('uploaded:', name, '-> /content/input.mp4')

## 4. Run Wav2Lip (lips → dubbed audio)

In [ ]:
!cd /content/Wav2Lip && ./.venv/bin/python inference.py \
  --checkpoint_path checkpoints/wav2lip_gan.pth \
  --face /content/input.mp4 --audio /content/input.mp4 \
  --outfile /content/lipsynced.mp4 --nosmooth
print('done -> /content/lipsynced.mp4')

## 5. (Optional) burn small subtitles at the bottom
Skip this cell if you don't want captions. Otherwise upload your `.srt` (e.g.
`fc4BozJTvc4.en.srt`) when prompted.

In [ ]:
from google.colab import files
up = files.upload()
srt = list(up.keys())[0]
style = "FontName=DejaVu Sans,FontSize=13,Alignment=2,MarginV=10,Outline=1,Shadow=0"
!ffmpeg -y -i /content/lipsynced.mp4 \
  -vf "subtitles={srt}:force_style='{style}'" \
  -c:a copy -movflags +faststart /content/lipsynced_subbed.mp4
print('done -> /content/lipsynced_subbed.mp4')

## 6. Preview & download

In [ ]:
import os
from IPython.display import Video, display
out = '/content/lipsynced_subbed.mp4' if os.path.exists('/content/lipsynced_subbed.mp4') else '/content/lipsynced.mp4'
print(out)
display(Video(out, embed=True, width=360))
from google.colab import files; files.download(out)